## Part 1: Preprocessing

In [1]:
# Import our dependencies
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras import layers

#  Import and read the attrition data
attrition_df = pd.read_csv('https://static.bc-edx.com/ai/ail-v-1-0/m19/lms/datasets/attrition.csv')
attrition_df.head()

2025-01-13 09:29:19.142774: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


,Age,Attrition,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,HourlyRate,JobInvolvement,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,Sales,1,2,Life Sciences,2,94,3,...,3,1,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,Research & Development,8,1,Life Sciences,3,61,2,...,4,4,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,Research & Development,2,2,Other,4,92,2,...,3,2,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,Research & Development,3,4,Life Sciences,4,56,3,...,3,3,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,Research & Development,2,1,Medical,1,40,3,...,3,4,1,6,3,3,2,2,2,2


In [2]:
# Determine the number of unique values in each column.
attrition_df.nunique()

Age                         43
Attrition                    2
BusinessTravel               3
Department                   3
DistanceFromHome            29
Education                    5
EducationField               6
EnvironmentSatisfaction      4
HourlyRate                  71
JobInvolvement               4
JobLevel                     5
JobRole                      9
JobSatisfaction              4
MaritalStatus                3
NumCompaniesWorked          10
OverTime                     2
PercentSalaryHike           15
PerformanceRating            2
RelationshipSatisfaction     4
StockOptionLevel             4
TotalWorkingYears           40
TrainingTimesLastYear        7
WorkLifeBalance              4
YearsAtCompany              37
YearsInCurrentRole          19
YearsSinceLastPromotion     16
YearsWithCurrManager        18
dtype: int64

In [3]:
# Create y_df with the Attrition and Department columns
y_df = attrition_df[['Attrition','Department']]
y_df.head()


,Attrition,Department
0,Yes,Sales
1,No,Research & Development
2,Yes,Research & Development
3,No,Research & Development
4,No,Research & Development


In [4]:
# Create a list of at least 10 column names to use as X data
X_columns_list = ['Age','DistanceFromHome','EnvironmentSatisfaction','HourlyRate','JobInvolvement','JobSatisfaction','TotalWorkingYears','WorkLifeBalance','YearsInCurrentRole','YearsSinceLastPromotion']

# Create X_df using your selected columns
X_df = attrition_df[X_columns_list]

# Show the data types for X_df
X_df.dtypes

Age                        int64
DistanceFromHome           int64
EnvironmentSatisfaction    int64
HourlyRate                 int64
JobInvolvement             int64
JobSatisfaction            int64
TotalWorkingYears          int64
WorkLifeBalance            int64
YearsInCurrentRole         int64
YearsSinceLastPromotion    int64
dtype: object

In [5]:
# Split the data into training and testing sets
from sklearn.model_selection import train_test_split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_df, y_df, random_state=78)

In [6]:
# Convert your X data to numeric data types however you see fit
# Add new code cells as necessary
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)
attrition_df["OverTime"].value_counts()

(1102, 10) (368, 10) (1102, 2) (368, 2)


OverTime
No     1054
Yes     416
Name: count, dtype: int64

In [7]:
# Create a StandardScaler
scaler = StandardScaler()

# Fit the StandardScaler to the training data
X_scaler = scaler.fit(X_train)

# Scale the training and testing data
X_train_scaled = X_scaler.transform(X_train)
X_test_scaled = X_scaler.transform(X_test)

In [ ]:
# Create a OneHotEncoder for the Department column
from sklearn.preprocessing import OneHotEncoder

ohe_dept = OneHotEncoder(sparse_output=False)

# Fit the encoder to the training data
ohe_dept.fit(y_train['Department'].values.reshape(-1,1))

# Create two new variables by applying the encoder
# to the training and testing data
train_dept_encoded = ohe_dept.transform(y_train['Department'].values.reshape(-1,1))
test_dept_encoded = ohe_dept.transform(y_test['Department'].values.reshape(-1,1))
train_dept_encoded

array([[0., 0., 1.],
       [0., 1., 0.],
       [0., 0., 1.],
       ...,
       [0., 1., 0.],
       [0., 0., 1.],
       [0., 1., 0.]])

In [9]:
# Create a OneHotEncoder for the Attrition column
ohe_attr = OneHotEncoder(sparse_output=False)

# Fit the encoder to the training data
ohe_attr.fit(y_train['Attrition'].values.reshape(-1,1))

# Create two new variables by applying the encoder
# to the training and testing data
train_attr_encoded = ohe_attr.transform(y_train['Attrition'].values.reshape(-1,1))
test_attr_encoded = ohe_attr.transform(y_test['Attrition'].values.reshape(-1,1))
train_attr_encoded

array([[0., 1.],
       [1., 0.],
       [0., 1.],
       ...,
       [1., 0.],
       [1., 0.],
       [0., 1.]])

## Create, Compile, and Train the Model

In [10]:
# Find the number of columns in the X training data
num_columns = X_train.shape[1]

# Create the input layer
input_layer = layers.Input(shape = (num_columns,))

# Create at least two shared layers
shared_layer1 = layers.Dense(64, activation='relu', name='Shared_1')(input_layer)
shared_layer2 = layers.Dense(32, activation='relu', name='Shared_2')(shared_layer1)

In [12]:
# Create a branch for Department
# with a hidden layer and an output layer

# Create the hidden layer
dept_hidden_layer1 = layers.Dense(32, activation='relu', name = 'dept_hidden')(shared_layer2)

# Create the output layer
dept_output_layer1 = layers.Dense(units=train_dept_encoded.shape[1], activation='softmax', name = 'dept_output')(dept_hidden_layer1)

In [14]:
# Create a branch for Attrition
# with a hidden layer and an output layer

# Create the hidden layer
attr_hidden_layer2 = layers.Dense(32, activation='relu', name = 'attr_hidden')(shared_layer2)

# Create the output layer
attr_output_layer2 = layers.Dense(units=train_attr_encoded.shape[1], activation='sigmoid', name = 'attr_output')(attr_hidden_layer2)

In [18]:
# Create the model
model = Model(inputs=input_layer, outputs=[dept_output_layer1, attr_output_layer2])

# Compile the model
model.compile(optimizer='adam',
              loss={'dept_output': 'categorical_crossentropy', 'attr_output': 'binary_crossentropy'},
              metrics={'dept_output':'accuracy', 'attr_output':'accuracy'})

# Summarize the model
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 10)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Shared_1 (Dense)    │ (None, 64)        │        704 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Shared_2 (Dense)    │ (None, 32)        │      2,080 │ Shared_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dept_hidden (Dense) │ (None, 32)        │      1,056 │ Shared_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attr_hidden (Dense) │ (None, 32)        │      1,056 │ Shared_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dept_output (Dense) │ (None, 3)         │         99 │ dept_hidden[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attr_output (Dense) │ (None, 2)         │         66 │ attr_hidden[0][0] │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,061 (19.77 KB)

 Trainable params: 5,061 (19.77 KB)

 Non-trainable params: 0 (0.00 B)

In [19]:
# Train the model
model.fit(X_train_scaled, {'dept_output': train_dept_encoded, 'attr_output': train_attr_encoded}, epochs=100, shuffle=True, verbose=2)


Epoch 1/100
35/35 - 5s - 130ms/step - attr_output_accuracy: 0.7069 - attr_output_loss: 0.6393 - dept_output_accuracy: 0.5508 - dept_output_loss: 0.9792 - loss: 1.6196
Epoch 2/100
35/35 - 0s - 7ms/step - attr_output_accuracy: 0.8276 - attr_output_loss: 0.4810 - dept_output_accuracy: 0.6661 - dept_output_loss: 0.7909 - loss: 1.2700
Epoch 3/100
35/35 - 0s - 6ms/step - attr_output_accuracy: 0.8276 - attr_output_loss: 0.4394 - dept_output_accuracy: 0.6661 - dept_output_loss: 0.7650 - loss: 1.2063
Epoch 4/100
35/35 - 0s - 6ms/step - attr_output_accuracy: 0.8276 - attr_output_loss: 0.4288 - dept_output_accuracy: 0.6661 - dept_output_loss: 0.7627 - loss: 1.1823
Epoch 5/100
35/35 - 0s - 6ms/step - attr_output_accuracy: 0.8348 - attr_output_loss: 0.4161 - dept_output_accuracy: 0.6661 - dept_output_loss: 0.7535 - loss: 1.1658
Epoch 6/100
35/35 - 0s - 7ms/step - attr_output_accuracy: 0.8394 - attr_output_loss: 0.4104 - dept_output_accuracy: 0.6652 - dept_output_loss: 0.7455 - loss: 1.1516
Epoch 7/

In [20]:
# Evaluate the model with the testing data
results_test = model.evaluate(X_test_scaled, {'dept_output': test_dept_encoded, 'attr_output': test_attr_encoded}, verbose=2)
print(results_test)

12/12 - 1s - 111ms/step - attr_output_accuracy: 0.8315 - attr_output_loss: 0.6898 - dept_output_accuracy: 0.5109 - dept_output_loss: 1.9002 - loss: 2.6328
[2.6328132152557373, 1.900206446647644, 0.6898112297058105, 0.83152174949646, 0.510869562625885]


In [23]:
# Print the accuracy for both department and attrition
print(f"Department predictions Accuracy: {results_test[4]}")
print(f"Attrition predictions Accuracy: {results_test[3]}")

Department predictions Accuracy: 0.510869562625885
Attrition predictions Accuracy: 0.83152174949646


# Summary

In the provided space below, briefly answer the following questions.

1. Is accuracy the best metric to use on this data? Why or why not?

2. What activation functions did you choose for your output layers, and why?

3. Can you name a few ways that this model might be improved?

1. Our results show Dept accuracy at 51% and Attrition accuracy at 83%.  While these results are decent, accuracy is not always the best metric. But for now, we could work with accuracy, perhaps by adding more layers. 
2. I chose the activation function 'softmax' for Department output and 'sigmoid' for Attrition output.  Softmax is ideal for the Department output since softmax is multiclass, and there were three departments.  Those three departments could be assigned values between 0 & 1 based on the classification of which is the "max".  Sigmoid is ideal for Attrition since there were only 2 values available (Yes or No) and could also be assigned values between 0 and 1. 
3. A few ways that this model might be improved:
a) Increase the number of epochs to further reduce loss and increase acuracy.
b) Add additional neural layers to again reduce loss and increase accuracy
c) Create additional features.  For example, calculate satisfaction scores to help improve predictions.